# Dự án DSS — Khám phá dữ liệu & Xác định thông số chính

**Mục tiêu của notebook này (bước trước khi huấn luyện mô hình):**

1. Đọc toàn bộ 12 bảng dữ liệu trong thư mục `Data_cleaned`.
2. Kiểm tra cấu trúc, kiểu dữ liệu, khóa liên kết giữa các bảng.
3. Phân tích **giá trị thiếu (null)** và **giá trị đã có (non-null)** cho từng bảng, để biết bảng nào đã sạch, bảng nào cần xử lý thêm.
4. Thống kê mô tả, kiểm tra lại outlier, phân tích tương quan cho các biến số quan trọng.
5. Xây dựng pipeline tiền xử lý **tổng quát** (`SimpleImputer` + `OneHotEncoder`) — pipeline này hoạt động đúng **dù dữ liệu có null hay không có null**, để bước huấn luyện mô hình sau này không bị lỗi khi dữ liệu thay đổi (ví dụ sau khi join nhiều bảng).
6. Tổng hợp các "thông số chính" (số dòng, số cột, tỉ lệ thiếu, biến mục tiêu khả thi...) của từng bảng, làm cơ sở chọn bài toán mô hình hóa cụ thể ở bước tiếp theo.

> Cấu trúc phân tích được tham khảo theo cách trình bày của tài liệu khóa trước (`ref.pdf`, làm trên dữ liệu nhà đất Miami), áp dụng lại cho bộ dữ liệu ngân hàng + CRM + bất động sản hiện tại.


## 0. Kết nối Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Sửa 'DSS_Project' thành tên thư mục bạn tạo trên Drive chứa dữ liệu
    BASE_DIR = '/content/drive/MyDrive/DSS_Project'
    print('Đã kết nối Google Drive thành công!')
except ImportError:
    BASE_DIR = '.'
    print('Không ở trong môi trường Colab, sử dụng đường dẫn local.')


In [ ]:
import os

DATA_DIR = os.path.join(BASE_DIR, 'Data_cleaned')
print(f'Thư mục dữ liệu: {DATA_DIR}\n')
print('Các file trong thư mục:')
for f in sorted(os.listdir(DATA_DIR)):
    print(' -', f)


## 0.1. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)


## 1. Đọc toàn bộ các bảng dữ liệu

Toàn bộ 12 file CSV trong `Data_cleaned` được đọc vào một dictionary `dfs` để tiện xử lý theo vòng lặp ở các bước sau (thay vì viết lặp lại code cho từng bảng).

In [ ]:
TABLE_FILES = {
    'account':         'completedacct.csv',
    'card':             'completedcard.csv',
    'client':           'completedclient.csv',
    'disposition':      'completeddisposition.csv',
    'district':         'completeddistrict.csv',
    'loan':             'completedloan.csv',
    'order':            'completedorder.csv',
    'trans':            'completedtrans.csv',
    'crm_call_center':  'crm_call_center_logs.csv',
    'crm_events':       'crm_events.csv',
    'crm_reviews':      'crm_reviews.csv',
    'luxury_loan':      'luxuryloanportfolio.csv',
}

dfs = {}
print('1. ĐỌC DỮ LIỆU TỪ TẤT CẢ CÁC BẢNG\n')
for name, filename in TABLE_FILES.items():
    path = os.path.join(DATA_DIR, filename)
    df = pd.read_csv(path)
    dfs[name] = df
    print(f'  - {name:16s} ({filename:26s}): {df.shape[0]:>9,} dòng x {df.shape[1]:>3} cột')


## 2. Xem nhanh cấu trúc từng bảng (kiểu dữ liệu + vài dòng đầu)

In [ ]:
def quick_look(name, df, n=3):
    print(f'\n{"="*90}\nBảng: {name}   |   shape = {df.shape}\n{"="*90}')
    print(df.dtypes)
    print('\nMẫu dữ liệu:')
    display(df.head(n))

for name, df in dfs.items():
    quick_look(name, df)


## 3. Phân tích giá trị thiếu (null) và giá trị đã có (non-null)

Đây là phần trả lời trực tiếp câu hỏi "giải quyết cả null lẫn non-null": hàm `missing_report` hoạt động đúng với **mọi** bảng, dù bảng đó có null hay hoàn toàn không có null — không cần biết trước tình trạng dữ liệu.

In [ ]:
def missing_report(df):
    n = len(df)
    miss = df.isnull().sum()
    pct = (miss / n * 100).round(2)
    report = pd.DataFrame({
        'non_null': n - miss,
        'null_count': miss,
        'null_pct': pct,
        'dtype': df.dtypes
    })
    return report.sort_values('null_count', ascending=False)

print('3. TỔNG HỢP GIÁ TRỊ THIẾU CHO TỪNG BẢNG\n')
summary_rows = []
for name, df in dfs.items():
    rep = missing_report(df)
    total_missing = int(rep['null_count'].sum())
    cols_with_missing = int((rep['null_count'] > 0).sum())
    summary_rows.append({
        'table': name,
        'n_rows': len(df),
        'n_cols': df.shape[1],
        'total_missing_cells': total_missing,
        'cols_with_missing': cols_with_missing
    })
    if total_missing > 0:
        print(f'--- {name}: có {cols_with_missing} cột bị thiếu dữ liệu ---')
        display(rep[rep['null_count'] > 0])
    else:
        print(f'--- {name}: KHÔNG có giá trị thiếu (toàn bộ {len(df):,} dòng đều non-null) ---')

missing_summary = pd.DataFrame(summary_rows)
print('\nTổng hợp tình trạng thiếu dữ liệu trên toàn bộ các bảng:')
display(missing_summary)


## 4. Thống kê mô tả cho các biến số

In [ ]:
print('4. THỐNG KÊ MÔ TẢ (CÁC BIẾN SỐ) CHO TỪNG BẢNG\n')
for name, df in dfs.items():
    numeric_df = df.select_dtypes(include=[np.number])
    if numeric_df.shape[1] == 0:
        print(f'--- {name}: không có biến số ---\n')
        continue
    print(f'--- {name} ---')
    display(numeric_df.describe().T)


## 5. Tự động phân loại biến: numeric / categorical / id / datetime

Nhiều cột tên `*_id`, `is_*`, `has_*` có kiểu `int64` nhưng về bản chất là **định danh** hoặc **phân loại**, không nên đưa thẳng vào mô hình như biến số liên tục. Hàm dưới đây tự phân loại theo tên cột và đặc điểm dữ liệu — dùng chung được cho mọi bảng.

In [ ]:
def classify_columns(df, max_cat_unique=20):
    numeric_cols, categorical_cols, id_cols, datetime_like_cols = [], [], [], []
    for col in df.columns:
        col_lower = col.lower()
        if col_lower.endswith('_id') or col_lower == 'id':
            id_cols.append(col)
            continue
        if 'date' in col_lower or col_lower in ('fulltime',):
            datetime_like_cols.append(col)
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            if df[col].nunique() <= max_cat_unique and col_lower.startswith(('is_', 'has_')):
                categorical_cols.append(col)
            else:
                numeric_cols.append(col)
        else:
            categorical_cols.append(col)
    return {
        'numeric': numeric_cols,
        'categorical': categorical_cols,
        'id': id_cols,
        'datetime_like': datetime_like_cols
    }

print('5. PHÂN LOẠI BIẾN CHO TỪNG BẢNG\n')
column_types = {}
for name, df in dfs.items():
    ctypes = classify_columns(df)
    column_types[name] = ctypes
    print(f'--- {name} ---')
    for k, v in ctypes.items():
        print(f'  {k:14s}: {v}')
    print()


## 6. Kiểm tra lại outlier (IQR) cho các biến tiền/giá trị quan trọng

Nhiều bảng (`loan`, `order`, `trans`, `luxury_loan`) đã có sẵn các cột `*_is_outlier`, `*_capped`, `*_log1p` — nghĩa là việc xử lý outlier đã được thực hiện từ trước. Ở đây ta **tính lại** bằng phương pháp IQR (giống `ref.pdf`) để kiểm chứng và so sánh với cờ đã có sẵn.

In [ ]:
def iqr_outlier_count(series):
    s = series.dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return int(((s < lower) | (s > upper)).sum()), lower, upper

print('6. KIỂM TRA LẠI OUTLIER (IQR) CHO CÁC BIẾN QUAN TRỌNG\n')
key_amount_columns = {
    'loan':        ['amount', 'payments'],
    'order':       ['amount'],
    'trans':       ['amount', 'balance'],
    'luxury_loan': ['funded_amount', 'payments', 'loan_balance', 'property_value'],
}

for table_name, cols in key_amount_columns.items():
    df = dfs[table_name]
    print(f'--- Bảng {table_name} ---')
    for col in cols:
        n_out, lower, upper = iqr_outlier_count(df[col])
        existing_flag_col = f'{col}_is_outlier'
        note = ''
        if existing_flag_col in df.columns:
            note = f'  (cờ có sẵn trong dữ liệu: {int(df[existing_flag_col].sum())} dòng được đánh dấu outlier)'
        print(f'  {col:18s}: {n_out:>7,} outlier theo IQR   [bound: {lower:,.1f} -> {upper:,.1f}]{note}')
    print()


## 7. Trực quan hóa phân phối cho vài biến mục tiêu tiềm năng

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.histplot(dfs['loan']['amount'], kde=True, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Phân phối: loan.amount')

sns.histplot(dfs['luxury_loan']['funded_amount'], kde=True, ax=axes[0, 1], color='darkorange')
axes[0, 1].set_title('Phân phối: luxury_loan.funded_amount')

sns.histplot(dfs['luxury_loan']['property_value'], kde=True, ax=axes[1, 0], color='seagreen')
axes[1, 0].set_title('Phân phối: luxury_loan.property_value')

sns.histplot(dfs['trans']['amount'].sample(50000, random_state=42), kde=True, ax=axes[1, 1], color='indianred')
axes[1, 1].set_title('Phân phối: trans.amount (mẫu 50,000 dòng)')

plt.tight_layout()
plt.show()


## 8. Phân tích tương quan giữa các biến số

In [ ]:
def plot_corr(df, numeric_cols, title, figsize=(8, 6)):
    plt.figure(figsize=figsize)
    corr = df[numeric_cols].corr()
    sns.heatmap(corr, annot=True, fmt='.1f', cmap='coolwarm', center=0)
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_corr(dfs['loan'], ['amount', 'duration', 'payments'], 'Ma trận tương quan - loan')

plot_corr(
    dfs['luxury_loan'],
    ['funded_amount', 'interest_rate_percent', 'payments', 'loan_balance',
     'property_value', 'total_units', 'gross_square_feet'],
    'Ma trận tương quan - luxury_loan'
)


## 9. Kiểm tra khóa liên kết & tính nhất quán giữa các bảng

Dữ liệu này có cấu trúc quan hệ (account, client, disposition, card, loan, order, trans cùng thuộc một "ngân hàng ảo"; crm_* là dữ liệu CRM riêng; luxury_loan là bộ dữ liệu bất động sản độc lập). Trước khi join bất kỳ bảng nào với nhau, cần kiểm tra kiểu dữ liệu của cột khóa có khớp nhau không.

In [ ]:
print('9. KIỂM TRA KHÓA LIÊN KẾT GIỮA CÁC BẢNG\n')

key_columns_to_check = ['account_id', 'client_id', 'district_id', 'disp_id', 'card_id', 'loan_id']
for key in key_columns_to_check:
    print(f'--- Cột khóa: {key} ---')
    for name, df in dfs.items():
        if key in df.columns:
            print(f'  {name:16s}: dtype={str(df[key].dtype):8s}  n_unique={df[key].nunique():>8,}  n_rows={len(df):>8,}')
    print()

print('LƯU Ý: nếu dtype khác nhau giữa các bảng (ví dụ object vs int64) cho cùng 1 khóa,')
print('cần ép kiểu (cast) đồng nhất trước khi join/merge các bảng để tránh lỗi hoặc join sai.')


## 10. Xây dựng pipeline tiền xử lý tổng quát (xử lý được cả null và non-null)

Hàm `create_preprocessor` dưới đây dùng `SimpleImputer` cho cả biến số và biến phân loại — nếu dữ liệu **không có null**, imputer chỉ đơn giản không làm gì; nếu dữ liệu **có null**, imputer sẽ tự điền giá trị phù hợp. Nhờ vậy, pipeline này có thể tái sử dụng cho bất kỳ bảng/bài toán nào ở bước huấn luyện mô hình tiếp theo, không phụ thuộc vào việc dữ liệu đầu vào có sạch hay không.

In [ ]:
def create_preprocessor(numeric_features, categorical_features, scale_numeric=True):
    """
    Tạo bộ tiền xử lý tổng quát, dùng chung cho mọi bảng / mọi biến mục tiêu sau này.

    - Biến số: SimpleImputer(strategy='mean') -> (tuỳ chọn) StandardScaler.
    - Biến phân loại: SimpleImputer(strategy='most_frequent') -> OneHotEncoder(handle_unknown='ignore').
    - Cả 2 nhánh đều dùng Imputer nên pipeline hoạt động đúng dù dữ liệu CÓ null hay KHÔNG có null.
    """
    numeric_steps = [('imputer', SimpleImputer(strategy='mean'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    numeric_transformer = Pipeline(steps=numeric_steps)

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='drop'
    )
    return preprocessor

print('Đã định nghĩa hàm create_preprocessor() — sẽ tái sử dụng ở notebook huấn luyện mô hình.')


### 10.1. Thử nghiệm pipeline: trường hợp KHÔNG có null và trường hợp CÓ null (giả lập)

In [ ]:
# Demo trên bảng luxury_loan (chỉ để kiểm tra pipeline, chưa phải bước huấn luyện chính thức)
demo_df = dfs['luxury_loan'].copy()

demo_numeric = ['interest_rate_percent', 'payments', 'total_units', 'gross_square_feet',
                'land_square_feet', 'employment_length']
demo_categorical = ['purpose', 'tax_class_at_present', 'building_class_at_present']

# (a) Trường hợp dữ liệu gốc - không có null
prep_a = create_preprocessor(demo_numeric, demo_categorical)
X_a = prep_a.fit_transform(demo_df[demo_numeric + demo_categorical])
print(f'(a) Dữ liệu gốc (không null)      -> shape sau tiền xử lý: {X_a.shape}')

# (b) Giả lập có null để kiểm tra pipeline có "sống sót" không
demo_df_null = demo_df.copy()
demo_df_null.loc[0:5, 'interest_rate_percent'] = np.nan
demo_df_null.loc[0:5, 'purpose'] = np.nan

prep_b = create_preprocessor(demo_numeric, demo_categorical)
X_b = prep_b.fit_transform(demo_df_null[demo_numeric + demo_categorical])
print(f'(b) Dữ liệu giả lập có null       -> shape sau tiền xử lý: {X_b.shape}')

print('\n=> Pipeline create_preprocessor() chạy ổn định trong cả 2 trường hợp.')


## 11. Tổng hợp các "thông số chính" — cơ sở chọn bài toán mô hình hóa

Bảng dưới đây tổng hợp đặc điểm chính của các bảng có tiềm năng làm bài toán mô hình hóa, kèm gợi ý biến mục tiêu và loại bài toán (phân loại/hồi quy).

In [ ]:
key_parameters = pd.DataFrame([
    {
        'table': 'loan', 'n_rows': len(dfs['loan']), 'candidate_target': 'status',
        'task_type': 'Phân loại — dự đoán khả năng trả nợ / nợ xấu',
        'note': 'Có thể join với account, client, district để bổ sung đặc trưng'
    },
    {
        'table': 'trans', 'n_rows': len(dfs['trans']), 'candidate_target': 'type / operation (hoặc balance)',
        'task_type': 'Phân loại loại giao dịch / Hồi quy số dư',
        'note': 'Bảng lớn nhất (>1 triệu dòng) — nên lấy mẫu khi thử nghiệm mô hình'
    },
    {
        'table': 'luxury_loan', 'n_rows': len(dfs['luxury_loan']), 'candidate_target': 'property_value hoặc loan_balance',
        'task_type': 'Hồi quy — dự đoán giá trị BĐS / dư nợ vay',
        'note': 'Cấu trúc tương tự bài toán SALE_PRC trong ref.pdf'
    },
    {
        'table': 'crm_call_center', 'n_rows': len(dfs['crm_call_center']), 'candidate_target': 'outcome',
        'task_type': 'Phân loại — kết quả xử lý cuộc gọi khách hàng',
        'note': 'priority, type, server là các đặc trưng phân loại có sẵn'
    },
    {
        'table': 'crm_events', 'n_rows': len(dfs['crm_events']), 'candidate_target': 'company_response_to_consumer',
        'task_type': 'Phân loại — phản hồi của công ty với khiếu nại',
        'note': 'Liên kết được với client qua client_id (cần đồng bộ dtype trước khi join)'
    },
])
display(key_parameters)

print("""
GHI CHÚ QUAN TRỌNG TRƯỚC KHI HUẤN LUYỆN MÔ HÌNH:
1. Toàn bộ 12 bảng trong Data_cleaned hiện KHÔNG có giá trị null (đã được làm sạch sẵn từ trước).
2. Một số bảng (loan, order, trans, luxury_loan) đã có sẵn các cột *_is_outlier, *_capped, *_log1p
   -> có thể dùng trực tiếp các cột này thay vì phải tự xử lý outlier lại từ đầu.
3. Cột client_id có kiểu dữ liệu KHÔNG đồng nhất giữa các bảng (object ở client, int64 ở
   disposition/crm_events) -> cần ép kiểu đồng nhất trước khi join/merge.
4. Hàm create_preprocessor() ở mục 10 đã được kiểm tra hoạt động tốt với cả dữ liệu
   có null và không có null, có thể tái sử dụng cho bất kỳ bài toán nào chọn ở bước tiếp theo.
5. Bảng trans rất lớn (1,056,320 dòng) — nên cân nhắc lấy mẫu (sampling) khi thử nghiệm mô hình
   trước khi chạy trên toàn bộ dữ liệu.
""")


## 12. Bước tiếp theo

Sau khi đã nắm được các thông số chính của dữ liệu, bước tiếp theo sẽ là:

1. Chọn **một** bài toán mô hình hóa cụ thể (ví dụ: dự đoán nợ xấu từ bảng `loan`, hoặc dự đoán giá trị bất động sản từ `luxury_loan`).
2. Join/merge các bảng liên quan (nếu cần), sau khi đã đồng bộ kiểu dữ liệu của các cột khóa (mục 9).
3. Tách tập train/test (giống `train_test_split` trong `ref.pdf`), áp dụng `create_preprocessor()` đã xây ở mục 10.
4. Huấn luyện các mô hình cơ bản để làm baseline — ví dụ Logistic Regression / Random Forest cho bài toán phân loại, Linear Regression / Random Forest Regressor cho bài toán hồi quy.
